# DataPreprocessing — Vehicle Sales Price Prediction

Final fixed notebook. It avoids data leakage: train/test split is done before fitting preprocessing objects. The notebook uses only `pandas`, `numpy`, `scikit-learn`, and `joblib`; it does not use `category_encoders` or version-sensitive `OneHotEncoder` parameters.

In [1]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
import sklearn

RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET = 'sellingprice'
MIN_CATEGORY_FREQUENCY = 10
MAX_CATEGORIES_PER_FEATURE = 50

# Detect project root robustly for both cases:
# 1) notebook is in project root
# 2) notebook is in project/notebooks
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
MODELS_DIR = PROJECT_ROOT / 'models'

for directory in [DATA_DIR, PROCESSED_DIR, REPORTS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# The same notebook should work even if the CSV was uploaded to another common place.
candidate_paths = [
    PROJECT_ROOT / 'data' / 'car_prices_sample.csv',
    PROJECT_ROOT / 'car_prices_sample.csv',
    PROJECT_ROOT.parent / 'data' / 'car_prices_sample.csv',
    PROJECT_ROOT.parent / 'car_prices_sample.csv',
    Path.cwd().resolve() / 'data' / 'car_prices_sample.csv',
    Path.cwd().resolve() / 'car_prices_sample.csv',
    Path('/home/jovyan/__MANUAL/data/car_prices_sample.csv'),
    Path('/home/jovyan/__MANUAL/car_prices_sample.csv'),
    Path('/home/jovyan/data/car_prices_sample.csv'),
    Path('/home/jovyan/car_prices_sample.csv'),
]

# Extra fallback: search a few common roots recursively. This avoids path problems in JupyterLab.
for root in [PROJECT_ROOT, PROJECT_ROOT.parent, Path.cwd().resolve(), Path('/home/jovyan'), Path('/mnt/data')]:
    if root.exists():
        try:
            candidate_paths.extend(root.glob('**/car_prices_sample.csv'))
        except Exception:
            pass

# Keep order and remove duplicates.
seen_paths = set()
unique_candidate_paths = []
for p in candidate_paths:
    p = Path(p).resolve()
    if p not in seen_paths:
        seen_paths.add(p)
        unique_candidate_paths.append(p)
candidate_paths = unique_candidate_paths

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    checked = '\n'.join(str(p) for p in candidate_paths)
    raise FileNotFoundError(
        'Dataset car_prices_sample.csv was not found. Put it into data/car_prices_sample.csv.\n'
        f'Checked paths:\n{checked}'
    )

print('Project root:', PROJECT_ROOT)
print('Dataset path:', DATA_PATH)
print('scikit-learn version:', sklearn.__version__)

Project root: /home/jovyan/project_ivanov
Dataset path: /home/jovyan/project_ivanov/data/car_prices_sample.csv
scikit-learn version: 1.6.1


## 1. Load raw data

In [2]:
raw_df = pd.read_csv(DATA_PATH)
print('Raw dataset shape:', raw_df.shape)
print('Columns:', list(raw_df.columns))
raw_df.head()

Raw dataset shape: (50000, 15)
Columns: ['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state', 'condition', 'odometer', 'color', 'interior', 'seller', 'sellingprice', 'saledate']


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,sellingprice,saledate
0,2014,Kia,Optima,LX,Sedan,automatic,5xxgm4a71eg328703,tx,49.0,11485.0,gray,gray,kia motors america inc,15700,Wed Feb 04 2015 02:30:00 GMT-0800 (PST)
1,2014,Kia,Optima,LX,Sedan,automatic,5xxgm4a71eg318141,tx,26.0,14265.0,white,beige,kia motors america inc,15900,Wed Feb 04 2015 02:30:00 GMT-0800 (PST)
2,2006,Chrysler,Town and Country,Limited,Minivan,automatic,2a8gp64l96r821834,az,19.0,92649.0,gold,tan,onesource/southwest remarketing,4700,Thu Jan 22 2015 03:00:00 GMT-0800 (PST)
3,2006,Chevrolet,Silverado 1500,LS,Extended Cab,NaN,1gcek19bx6z269153,tx,22.0,166999.0,white,gray,texas direct auto,6700,Wed Feb 25 2015 02:20:00 GMT-0800 (PST)
4,2013,Chevrolet,Black Diamond Avalanche,LT,Crew Cab,automatic,3gntkfe78dg263779,tn,43.0,29265.0,—,black,north lily auto sales llc,32250,Wed Jan 28 2015 02:30:00 GMT-0800 (PST)


## 2. Basic cleaning and train/test split

In [3]:
df = raw_df.copy()

# Normalize column names only at the source level: strip whitespace and lowercase.
df.columns = [str(c).strip().lower() for c in df.columns]

if TARGET not in df.columns:
    raise ValueError(f'Target column {TARGET!r} was not found. Available columns: {list(df.columns)}')

# Convert target to numeric and drop rows without target.
df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')
df = df.dropna(subset=[TARGET]).copy()

# For car price prediction, non-positive prices are not informative for regression.
df = df[df[TARGET] > 0].copy()

# Drop identifiers / leakage-prone / very high-cardinality fields that are not useful for a simple tabular model.
DROP_COLUMNS = [c for c in ['vin', 'seller', 'saledate'] if c in df.columns]

X_raw = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET].astype(float)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print('Dropped columns:', DROP_COLUMNS)
print('Train raw:', X_train_raw.shape)
print('Test raw:', X_test_raw.shape)
print('Target train:', y_train.shape)
print('Target test:', y_test.shape)

Dropped columns: ['vin', 'seller', 'saledate']
Train raw: (40000, 11)
Test raw: (10000, 11)
Target train: (40000,)
Target test: (10000,)


## 3. Fit preprocessing on train and transform train/test

In [4]:
def make_unique_columns(columns):
    """Make column names unique while preserving order."""
    counts = {}
    result = []
    for col in map(str, columns):
        if col not in counts:
            counts[col] = 0
            result.append(col)
        else:
            counts[col] += 1
            result.append(f'{col}__dup{counts[col]}')
    return result


def safe_string_series(s: pd.Series, fill_value: str) -> pd.Series:
    """Return a string Series without missing values and without mixed object types."""
    return s.fillna(fill_value).astype(str)


def limit_categories(s: pd.Series, allowed_categories: list) -> pd.Series:
    """Keep known frequent categories; map rare/unseen values to __OTHER__."""
    allowed = set(map(str, allowed_categories))
    s = s.astype(str)
    return s.where(s.isin(allowed), '__OTHER__')


def fit_preprocess_train(X_train_raw: pd.DataFrame):
    X = X_train_raw.copy()
    X.columns = make_unique_columns(X.columns)

    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    numeric_medians = {}
    numeric_means = {}
    numeric_stds = {}
    numeric_parts = []

    for col in numeric_cols:
        values = pd.to_numeric(X[col], errors='coerce')
        median = values.median()
        if pd.isna(median):
            median = 0.0
        filled = values.fillna(median).astype(float)
        mean = filled.mean()
        std = filled.std(ddof=0)
        if pd.isna(std) or std == 0:
            std = 1.0
        scaled = (filled - mean) / std
        out_col = f'num__{col}'
        numeric_parts.append(pd.DataFrame({out_col: scaled}, index=X.index))
        numeric_medians[col] = float(median)
        numeric_means[col] = float(mean)
        numeric_stds[col] = float(std)

    cat_modes = {}
    allowed_categories = {}
    cat_parts = []

    for col in categorical_cols:
        mode = X[col].dropna().mode()
        fill_value = str(mode.iloc[0]) if len(mode) else 'missing'
        cat_modes[col] = fill_value

        s = safe_string_series(X[col], fill_value)
        counts = s.value_counts(dropna=False)
        allowed = counts[counts >= MIN_CATEGORY_FREQUENCY].head(MAX_CATEGORIES_PER_FEATURE).index.astype(str).tolist()
        if fill_value not in allowed:
            allowed = [fill_value] + allowed
        allowed = list(dict.fromkeys(allowed))
        allowed_categories[col] = allowed

        s_limited = limit_categories(s, allowed)
        dummies = pd.get_dummies(s_limited, prefix=f'cat__{col}', prefix_sep='__', dtype=float)
        cat_parts.append(dummies)

    parts = numeric_parts + cat_parts
    if parts:
        X_processed = pd.concat(parts, axis=1)
    else:
        X_processed = pd.DataFrame(index=X.index)

    if X_processed.columns.duplicated().any():
        X_processed = X_processed.T.groupby(level=0, sort=False).sum().T

    X_processed = X_processed.astype(float)
    feature_columns = list(dict.fromkeys(X_processed.columns.astype(str).tolist()))
    X_processed = X_processed.reindex(columns=feature_columns, fill_value=0.0)

    fitted = {
        'raw_columns': X.columns.tolist(),
        'numeric_cols': numeric_cols,
        'categorical_cols': categorical_cols,
        'numeric_medians': numeric_medians,
        'numeric_means': numeric_means,
        'numeric_stds': numeric_stds,
        'cat_modes': cat_modes,
        'allowed_categories': allowed_categories,
        'feature_columns': feature_columns,
        'min_category_frequency': MIN_CATEGORY_FREQUENCY,
        'max_categories_per_feature': MAX_CATEGORIES_PER_FEATURE,
    }
    return X_processed, fitted


def transform_preprocess_test(X_raw: pd.DataFrame, fitted: dict) -> pd.DataFrame:
    X = X_raw.copy()
    X.columns = make_unique_columns(X.columns)

    for col in fitted['raw_columns']:
        if col not in X.columns:
            X[col] = np.nan
    X = X[fitted['raw_columns']]

    parts = []

    for col in fitted['numeric_cols']:
        values = pd.to_numeric(X[col], errors='coerce')
        filled = values.fillna(fitted['numeric_medians'][col]).astype(float)
        scaled = (filled - fitted['numeric_means'][col]) / fitted['numeric_stds'][col]
        parts.append(pd.DataFrame({f'num__{col}': scaled}, index=X.index))

    for col in fitted['categorical_cols']:
        fill_value = fitted['cat_modes'][col]
        allowed = fitted['allowed_categories'][col]
        s = safe_string_series(X[col], fill_value)
        s_limited = limit_categories(s, allowed)
        dummies = pd.get_dummies(s_limited, prefix=f'cat__{col}', prefix_sep='__', dtype=float)
        parts.append(dummies)

    if parts:
        X_processed = pd.concat(parts, axis=1)
    else:
        X_processed = pd.DataFrame(index=X.index)

    if X_processed.columns.duplicated().any():
        X_processed = X_processed.T.groupby(level=0, sort=False).sum().T

    feature_columns = list(dict.fromkeys(fitted['feature_columns']))
    X_processed = X_processed.reindex(columns=feature_columns, fill_value=0.0)
    X_processed = X_processed.astype(float)
    return X_processed

X_train, fitted_preprocessor = fit_preprocess_train(X_train_raw)
X_test = transform_preprocess_test(X_test_raw, fitted_preprocessor)

print('Numerical features:', fitted_preprocessor['numeric_cols'])
print('Categorical features:', fitted_preprocessor['categorical_cols'])
print('Processed train:', X_train.shape)
print('Processed test:', X_test.shape)

assert X_train.shape[1] == X_test.shape[1], 'Train and test must have the same number of features'
assert list(X_train.columns) == list(X_test.columns), 'Train and test feature columns must match'
assert len(X_train) == len(y_train), 'X_train and y_train lengths must match'
assert len(X_test) == len(y_test), 'X_test and y_test lengths must match'
assert not X_train.columns.duplicated().any(), 'X_train has duplicated columns'
assert not X_test.columns.duplicated().any(), 'X_test has duplicated columns'
assert np.isfinite(X_train.to_numpy()).all(), 'X_train contains NaN or infinite values'
assert np.isfinite(X_test.to_numpy()).all(), 'X_test contains NaN or infinite values'

X_train.head()

Numerical features: ['year', 'condition', 'odometer']
Categorical features: ['make', 'model', 'trim', 'body', 'transmission', 'state', 'color', 'interior']
Processed train: (40000, 273)
Processed test: (10000, 273)


,num__year,num__condition,num__odometer,cat__make__Acura,cat__make__Audi,cat__make__BMW,cat__make__Buick,cat__make__Cadillac,cat__make__Chevrolet,cat__make__Chrysler,...,cat__interior__gray,cat__interior__green,cat__interior__off-white,cat__interior__orange,cat__interior__purple,cat__interior__red,cat__interior__silver,cat__interior__tan,cat__interior__white,cat__interior__—
39087,0.488534,1.367321,-0.545927,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30893,0.488534,-2.095776,-0.227505,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
45278,0.996861,0.313335,-0.758000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16398,-1.290610,0.238050,2.471686,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13653,-1.036446,-0.891220,1.113614,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Save processed data and preprocessing objects

In [5]:
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_train.to_frame(name=TARGET).to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_frame(name=TARGET).to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

# Optional raw splits for debugging/reproducibility.
X_train_raw.to_csv(PROCESSED_DIR / 'X_train_raw.csv', index=False)
X_test_raw.to_csv(PROCESSED_DIR / 'X_test_raw.csv', index=False)
y_train.to_frame(name=TARGET).to_csv(PROCESSED_DIR / 'y_train_raw.csv', index=False)
y_test.to_frame(name=TARGET).to_csv(PROCESSED_DIR / 'y_test_raw.csv', index=False)

joblib.dump(fitted_preprocessor, MODELS_DIR / 'preprocessor.joblib')

metadata = {
    'target': TARGET,
    'drop_columns': DROP_COLUMNS,
    'numeric_features': fitted_preprocessor['numeric_cols'],
    'categorical_features': fitted_preprocessor['categorical_cols'],
    'feature_count': int(X_train.shape[1]),
    'train_rows': int(X_train.shape[0]),
    'test_rows': int(X_test.shape[0]),
    'test_size': TEST_SIZE,
    'random_state': RANDOM_STATE,
    'sklearn_version': sklearn.__version__,
    'data_path': str(DATA_PATH),
    'min_category_frequency': MIN_CATEGORY_FREQUENCY,
    'max_categories_per_feature': MAX_CATEGORIES_PER_FEATURE,
}
with open(MODELS_DIR / 'preprocessing_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('Saved processed data to:', PROCESSED_DIR)
print('Saved preprocessor to:', MODELS_DIR / 'preprocessor.joblib')
print('Saved metadata to:', MODELS_DIR / 'preprocessing_metadata.json')

Saved processed data to: /home/jovyan/project_ivanov/data/processed
Saved preprocessor to: /home/jovyan/project_ivanov/models/preprocessor.joblib
Saved metadata to: /home/jovyan/project_ivanov/models/preprocessing_metadata.json


## 5. Quick checks

In [6]:
print('Missing values in X_train:', int(X_train.isna().sum().sum()))
print('Missing values in X_test:', int(X_test.isna().sum().sum()))
print('Duplicate columns in X_train:', int(X_train.columns.duplicated().sum()))
print('Duplicate columns in X_test:', int(X_test.columns.duplicated().sum()))
print('Saved files:')
for p in sorted(PROCESSED_DIR.glob('*.csv')):
    print('-', p.name)

print('\nPreprocessing finished successfully.')

Missing values in X_train: 0
Missing values in X_test: 0
Duplicate columns in X_train: 0
Duplicate columns in X_test: 0
Saved files:
- X_test.csv
- X_test_raw.csv
- X_train.csv
- X_train_raw.csv
- y_test.csv
- y_test_raw.csv
- y_train.csv
- y_train_raw.csv

Preprocessing finished successfully.
